# Baseline and Prompt Engineering

## 1. Цель этапа

На этом этапе строится базовая версия медицинского ассистента на `Qwen2.5-3B-Instruct` без RAG и fine-tuning.

Сравниваются два варианта:

- **A — Base Qwen + исходный system prompt**;
- **B — Base Qwen + улучшенный system prompt**.

Задача этапа — получить воспроизводимый baseline и отдельно проверить, насколько изменение system prompt влияет на ответы модели.

Для проверки кода используется `debug`, для сравнения вариантов — `dev`. Замороженный `test` на этом этапе не используется.

Во всех экспериментах модель и параметры генерации остаются одинаковыми; меняется только system prompt.

## 2. Tokenizer и подготовка входа модели

In [ ]:
import pandas as pd

import torch

from transformers import (
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    AutoTokenizer,
)

In [2]:
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
CUDA version: 13.0
GPU: NVIDIA GeForce RTX 4060


In [3]:
debug_path = "../materials/debug.csv"

debug_df = pd.read_csv(debug_path)

print(debug_df.shape)
debug_df.head()

(30, 2)


,input,output
0,I am a 48 year old male who had hip replacemen...,"Hello, The blood pressure reading you have men..."
1,Hi doc! I had all my blood tests done. Sugar f...,"Hi, dear. I have gone through your question. I..."
2,My mother (aged 65) is suffering from Hypothyr...,Hi. Thanks for your query. Read and understood...
3,I fell from a stool while standing on my tip t...,Hello have studied your case history. Traumati...
4,"Hi,i am femal 23 years old my wight 52kg ,leng...",Tithe cause of hair loss should be identified....


In [4]:
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

### Проверка токенизации

Перед подачей текста в модель tokenizer разбивает его на токены и преобразует каждый токен в числовой `token ID`.

Ниже этот переход проверяется на одном вопросе из `debug`.

In [5]:
question = debug_df["input"].iloc[0]

tokens = tokenizer.tokenize(question)
token_ids = tokenizer.encode(
    question,
    add_special_tokens=False,
)

print("Question:", question)
print("\nКоличество токенов:", len(tokens))
print("Первые токены:", tokens[:15])
print("Первые token IDs:", token_ids[:15])

Question: I am a 48 year old male who had hip replacement surgery 6 days ago.  I am taking blood pressure meds but my current readings over the past few days are on the high side (today was 138/97 pulse 77).  How does surgery and recovery impact BP?  Should I request my medication to be adjusted?

Количество токенов: 74
Первые токены: ['I', 'Ġam', 'Ġa', 'Ġ', '4', '8', 'Ġyear', 'Ġold', 'Ġmale', 'Ġwho', 'Ġhad', 'Ġhip', 'Ġreplacement', 'Ġsurgery', 'Ġ']
Первые token IDs: [40, 1079, 264, 220, 19, 23, 1042, 2310, 8593, 879, 1030, 18143, 13723, 14829, 220]


In [6]:
decoded_text = tokenizer.decode(token_ids)

assert decoded_text == question
print("Encode/decode sanity check passed.")

Encode/decode sanity check passed.


### Формат диалога

Instruction-модель получает диалог в виде последовательности ролей:

- `system` — задаёт общие правила поведения модели;
- `user` — содержит вопрос пользователя;
- `assistant` — содержит ответ модели.

Сначала формируется список `messages`, затем `chat template` переводит его в специальный формат Qwen со служебными токенами.

Во время генерации ответа `assistant` ещё отсутствует, поэтому `add_generation_prompt=True` добавляет начало нового assistant-turn и показывает модели, откуда нужно продолжать генерацию.

In [7]:
system_prompt = "You are a medical assistant."

messages = [
    {
        "role": "system",
        "content": system_prompt,
    },
    {
        "role": "user",
        "content": question,
    },
]

In [8]:
formatted_prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

without_generation_prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=False,
)

In [9]:
print("WITH generation prompt:\n")
print(formatted_prompt)

print("\n" + "=" * 80 + "\n")

print("WITHOUT generation prompt:\n")
print(without_generation_prompt)

WITH generation prompt:

<|im_start|>system
You are a medical assistant.<|im_end|>
<|im_start|>user
I am a 48 year old male who had hip replacement surgery 6 days ago.  I am taking blood pressure meds but my current readings over the past few days are on the high side (today was 138/97 pulse 77).  How does surgery and recovery impact BP?  Should I request my medication to be adjusted?<|im_end|>
<|im_start|>assistant



WITHOUT generation prompt:

<|im_start|>system
You are a medical assistant.<|im_end|>
<|im_start|>user
I am a 48 year old male who had hip replacement surgery 6 days ago.  I am taking blood pressure meds but my current readings over the past few days are on the high side (today was 138/97 pulse 77).  How does surgery and recovery impact BP?  Should I request my medication to be adjusted?<|im_end|>



### Подготовка входа модели

После применения `chat template` текст преобразуется в PyTorch tensors.

`input_ids` содержат идентификаторы токенов, а `attention_mask` показывает, какие позиции относятся к реальному тексту, а какие при необходимости являются padding.

In [10]:
inputs = tokenizer(
    formatted_prompt,
    return_tensors="pt",
    add_special_tokens=False,
)

In [11]:
print("input_ids:", inputs["input_ids"].shape)
print("attention_mask:", inputs["attention_mask"].shape)

print("\nПервые token IDs:")
print(inputs["input_ids"][0, :20])

input_ids: torch.Size([1, 93])
attention_mask: torch.Size([1, 93])

Первые token IDs:
tensor([151644,   8948,    198,   2610,    525,    264,   6457,  17847,     13,
        151645,    198, 151644,    872,    198,     40,   1079,    264,    220,
            19,     23])


In [12]:
print("Все позиции активны:", inputs["attention_mask"].all().item())

Все позиции активны: True


## 3. Загрузка Qwen2.5-3B-Instruct

Для baseline используется `Qwen2.5-3B-Instruct` — decoder-only language model, которая генерирует текст, последовательно предсказывая следующий токен.

Модель загружается через `AutoModelForCausalLM`.

Чтобы уменьшить использование GPU-памяти, веса загружаются в 4-bit формате. Quantization здесь используется только как технический способ запустить модель на доступном GPU и не является отдельным экспериментальным фактором.

Во всех сравниваемых вариантах используется одна и та же конфигурация модели.

In [13]:
COMPUTE_DTYPE = (
    torch.bfloat16
    if torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    else torch.float16
)

print("Compute dtype:", COMPUTE_DTYPE)

Compute dtype: torch.bfloat16


In [14]:
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
    bnb_4bit_use_double_quant=True,
)

In [15]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    device_map="auto",
)

model.eval()

W0909 23:28:01.549000 11204 Lib\site-packages\torch\utils\flop_counter.py:113] triton not found; flop counting will not work for triton kernels
Loading weights: 100%|██████████| 434/434 [00:02<00:00, 160.30it/s]


Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 2048)
    (layers): ModuleList(
      (0-35): 36 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=True)
          (k_proj): Linear4bit(in_features=2048, out_features=256, bias=True)
          (v_proj): Linear4bit(in_features=2048, out_features=256, bias=True)
          (o_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear4bit(in_features=2048, out_features=11008, bias=False)
          (up_proj): Linear4bit(in_features=2048, out_features=11008, bias=False)
          (down_proj): Linear4bit(in_features=11008, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((2048,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((2048,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm

### Проверка архитектуры модели

После загрузки посмотрим основные параметры Qwen и свяжем их с архитектурой Transformer.

In [16]:
print("Architecture:", model.config.model_type)
print("Hidden size:", model.config.hidden_size)
print("Transformer layers:", model.config.num_hidden_layers)
print("Attention heads:", model.config.num_attention_heads)
print("Key/Value heads:", model.config.num_key_value_heads)
print("Intermediate size:", model.config.intermediate_size)
print("Vocabulary size:", model.config.vocab_size)

Architecture: qwen2
Hidden size: 2048
Transformer layers: 36
Attention heads: 16
Key/Value heads: 2
Intermediate size: 11008
Vocabulary size: 151936


Модель содержит 36 decoder-блоков и hidden size 2048. Qwen использует Grouped Query Attention (GQA): query-heads больше, чем key/value-heads, что уменьшает объём KV cache при генерации.  
`LM head` преобразует hidden state размерности 2048 в logits по 151 936 токенам vocabulary.

### Один проход через модель

Перед полной генерацией выполним один `forward pass`.

Модель получает всю входную последовательность и для каждой её позиции возвращает logits по всему vocabulary. Это позволяет напрямую увидеть результат работы Transformer до того, как `model.generate()` начнёт выполнять autoregressive generation.

In [17]:
inputs = {
    name: tensor.to(model.device)
    for name, tensor in inputs.items()
}

In [18]:
with torch.inference_mode():
    outputs = model(**inputs)

In [19]:
print("Logits shape:", outputs.logits.shape)

Logits shape: torch.Size([1, 93, 151936])


Размерность logits имеет вид:

`[batch size, sequence length, vocabulary size]`.

То есть для каждой позиции входной последовательности модель рассчитывает отдельный score для каждого возможного следующего токена.

## 4. Как модель выбирает следующий токен

После `forward pass` модель возвращает logits для каждой позиции последовательности.

Для продолжения генерации используются logits последней позиции — они показывают, какие токены модель считает наиболее подходящими для следующего шага.

После `softmax` logits преобразуются в распределение вероятностей по vocabulary.

In [20]:
last_token_logits = outputs.logits[:, -1, :]

print(last_token_logits.shape)

torch.Size([1, 151936])


In [21]:
next_token_probs = torch.softmax(
    last_token_logits.float(),
    dim=-1,
)

In [22]:
top_probs, top_ids = torch.topk(
    next_token_probs,
    k=10,
    dim=-1,
)

for token_id, probability in zip(
    top_ids[0],
    top_probs[0],
):
    token = tokenizer.decode([token_id.item()])

    print(
        repr(token),
        "-",
        round(probability.item(), 4),
    )

'It' - 0.7122
'After' - 0.1092
'The' - 0.0585
'S' - 0.0585
'Post' - 0.0167
'Following' - 0.009
'Given' - 0.0062
'Re' - 0.0054
'Your' - 0.0033
'Congratulations' - 0.0029


In [23]:
next_token_id = torch.argmax(
    last_token_logits,
    dim=-1,
)

next_token = tokenizer.decode(
    next_token_id.item(),
)

print("Следующий токен:", repr(next_token))

Следующий токен: 'It'


## 5. Генерация полного ответа

Теперь вместо одного следующего токена можно запустить полноценную генерацию.

`model.generate()` повторяет один и тот же процесс: выбирает следующий токен, добавляет его к последовательности и снова запускает модель. Генерация продолжается до stop token или достижения заданного ограничения длины.

Для baseline используется `do_sample=False`, то есть deterministic greedy decoding. Это уменьшает случайность и позволяет корректнее сравнивать разные system prompts.

In [24]:
with torch.inference_mode():
    output_ids = model.generate(
        **inputs,
        do_sample=False,
        max_new_tokens=512,
    )

In [25]:
print("Input shape:", inputs["input_ids"].shape)
print("Output shape:", output_ids.shape)

Input shape: torch.Size([1, 93])
Output shape: torch.Size([1, 470])


In [26]:
input_length = inputs["input_ids"].shape[1]

generated_ids = output_ids[:, input_length:]

In [27]:
answer = tokenizer.decode(
    generated_ids[0],
    skip_special_tokens=True,
).strip()

print(answer)

It's important to monitor your blood pressure after surgery, especially in the early recovery period, as it can sometimes fluctuate due to various factors including anesthesia, pain management, and stress of the surgical procedure.

Surgery and recovery can indeed affect blood pressure in several ways:

1. **Anesthesia**: General anesthesia can cause temporary changes in blood pressure, which may return to normal within a few hours or days.
2. **Pain Management**: Pain medications can sometimes lead to elevated blood pressure, particularly if you're not used to taking them regularly.
3. **Stress and Anxiety**: The stress and anxiety associated with surgery can also temporarily raise blood pressure.
4. **Dehydration and Inflammation**: Post-surgery, dehydration and inflammation can contribute to higher blood pressure levels.

Given that your blood pressure readings have been consistently on the high side (138/97) and you are already on blood pressure medication, it might be wise to disc

### Проверка генерации

Модель успешно формирует связный законченный ответ в заданном chat format.

Этот пример используется только для проверки работы pipeline. Качество baseline и влияние system prompt дальше будут оцениваться на нескольких примерах из `debug`, а затем систематически на `dev`.

### Функция генерации ответа

После проверки отдельных шагов объединим подготовку `messages`, применение `chat template`, токенизацию и `model.generate()` в одну функцию.

Функция принимает вопрос и system prompt, формирует вход модели и возвращает только сгенерированный ответ.

In [28]:
def generate_answer(
    question,
    system_prompt,
    model,
    tokenizer,
    max_new_tokens=512,
):
    messages = [
        {
            "role": "system",
            "content": system_prompt,
        },
        {
            "role": "user",
            "content": question,
        },
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        add_special_tokens=False,
    ).to(model.device)

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            do_sample=False,
            max_new_tokens=max_new_tokens,
        )

    generated_ids = output_ids[:, inputs["input_ids"].shape[1]:]

    answer = tokenizer.decode(
        generated_ids[0],
        skip_special_tokens=True,
    ).strip()

    return answer

Проверим функцию на том же примере из `debug`, чтобы убедиться, что после объединения шагов результат генерируется корректно.

In [29]:
answer = generate_answer(
    question=question,
    system_prompt=system_prompt,
    model=model,
    tokenizer=tokenizer,
)

print(answer)

It's important to monitor your blood pressure after surgery, especially in the early recovery period, as it can sometimes fluctuate due to various factors including anesthesia, pain management, and stress of the surgical procedure.

Surgery and recovery can indeed affect blood pressure in several ways:

1. **Anesthesia**: General anesthesia can cause temporary changes in blood pressure, which may return to normal within a few hours or days.
2. **Pain Management**: Pain medications can sometimes lead to elevated blood pressure, particularly if you're not used to taking them regularly.
3. **Stress and Anxiety**: The stress and anxiety associated with surgery can also temporarily raise blood pressure.
4. **Dehydration and Inflammation**: Post-surgery, dehydration and inflammation can contribute to higher blood pressure levels.

Given that your blood pressure readings have been consistently on the high side (138/97) and you are already on blood pressure medication, it might be wise to disc

Функция корректно объединяет подготовку chat format, токенизацию и autoregressive generation.

Далее временный system prompt заменяется на baseline prompt, после чего сравниваются исходная и улучшенная инструкции.

## 6. Baseline с исходным system prompt

В качестве первого baseline используется короткий system prompt, задающий модели роль медицинского ассистента и требование отвечать на основе описания пациента.

Этот вариант служит контрольной точкой для дальнейшего сравнения с более детализированным prompt.

In [30]:
ORIGINAL_SYSTEM_PROMPT = (
    "If you are a doctor, please answer the medical questions "
    "based on the patient's description."
)

In [31]:
debug_examples = []

for idx, row in debug_df.head(5).iterrows():
    answer = generate_answer(
        question=row["input"],
        system_prompt=ORIGINAL_SYSTEM_PROMPT,
        model=model,
        tokenizer=tokenizer,
    )

    debug_examples.append(
        {
            "question": row["input"],
            "reference": row["output"],
            "baseline_answer": answer,
        }
    )

baseline_debug_df = pd.DataFrame(debug_examples)

In [32]:
for i, row in baseline_debug_df.iterrows():
    print(f"\nEXAMPLE {i + 1}")
    print("-" * 80)
    print("Question:", row["question"])
    print("\nBaseline:", row["baseline_answer"])


EXAMPLE 1
--------------------------------------------------------------------------------
Question: I am a 48 year old male who had hip replacement surgery 6 days ago.  I am taking blood pressure meds but my current readings over the past few days are on the high side (today was 138/97 pulse 77).  How does surgery and recovery impact BP?  Should I request my medication to be adjusted?

Baseline: It is important to monitor your blood pressure after surgery as it can sometimes change due to various factors including anesthesia, pain management, and stress of the surgical procedure. However, it's also common for blood pressure to fluctuate in the days following surgery as your body adjusts.

Your current blood pressure reading of 138/97 with a pulse rate of 77 is within a normal range, but it's still slightly elevated compared to pre-surgery levels or your usual baseline. It's not uncommon for blood pressure to rise post-surgery, especially if you were already on blood pressure medicati

### Наблюдения по baseline

На `debug` baseline формирует связные и релевантные ответы, однако проявляются повторяющиеся проблемы:

- излишне уверенные диагностические формулировки при недостатке данных;
- фактические ошибки в отдельных медицинских утверждениях;
- рекомендации, которые выходят за пределы информации, доступной из вопроса;
- недостаточное разделение между возможной причиной и установленным диагнозом;
- непоследовательное выражение неопределённости.

Поэтому improved system prompt должен в первую очередь усиливать осторожность формулировок, работу с неопределённостью и ограничения на неподтверждённые медицинские рекомендации.

## 7. Улучшенный system prompt

Анализ ответов baseline показал, что модель в целом отвечает связно и по теме, но иногда делает слишком уверенные медицинские выводы или даёт рекомендации, которые недостаточно обоснованы информацией из вопроса.

Поэтому новый system prompt должен:

- отделять возможные причины от установленного диагноза;
- явно учитывать недостаток информации;
- не придумывать симптомы, результаты обследований или другие факты;
- не рекомендовать самостоятельно начинать, отменять или менять дозировку лекарств;
- не назначать конкретные обследования без достаточных оснований;
- указывать на необходимость срочной помощи только при действительно тревожных признаках;
- давать понятный и практически полезный ответ без лишней категоричности.

In [33]:
IMPROVED_SYSTEM_PROMPT = """
You are a medical assistant providing general health information.

Use only the information given by the patient and medical knowledge you are confident about.

When answering:

1. Answer the main question directly.
2. Separate known facts from possible explanations.
3. If a diagnosis, laboratory result, vital sign, or medication effect cannot be interpreted confidently from the available information, explicitly say that rather than guessing.
4. Do not invent diagnoses, mechanisms, symptoms, test results, drug names, or medical history.
5. Do not introduce specific tests, treatments, or medication changes unless they are clearly necessary to answer the question.
6. Never advise starting, stopping, or changing medication without clinician supervision.
7. Mention urgent evaluation only when the information provided reasonably suggests a time-sensitive risk.
8. Keep the answer concise, practical, and appropriately cautious.

When important information is missing, state what is uncertain instead of filling the gap with assumptions.
""".strip()

In [34]:
improved_answers = []

for _, row in debug_df.head(5).iterrows():
    answer = generate_answer(
        question=row["input"],
        system_prompt=IMPROVED_SYSTEM_PROMPT,
        model=model,
        tokenizer=tokenizer,
    )

    improved_answers.append(answer)

In [35]:
baseline_debug_df["improved_answer"] = improved_answers

In [36]:
for i, row in baseline_debug_df.iterrows():
    print(f"\nEXAMPLE {i + 1}")
    print("=" * 80)

    print("\nQUESTION:\n")
    print(row["question"])

    print("\nORIGINAL PROMPT:\n")
    print(row["baseline_answer"])

    print("\nIMPROVED PROMPT:\n")
    print(row["improved_answer"])


EXAMPLE 1

QUESTION:

I am a 48 year old male who had hip replacement surgery 6 days ago.  I am taking blood pressure meds but my current readings over the past few days are on the high side (today was 138/97 pulse 77).  How does surgery and recovery impact BP?  Should I request my medication to be adjusted?

ORIGINAL PROMPT:

It is important to monitor your blood pressure after surgery as it can sometimes change due to various factors including anesthesia, pain management, and stress of the surgical procedure. However, it's also common for blood pressure to fluctuate in the days following surgery as your body adjusts.

Your current blood pressure reading of 138/97 with a pulse rate of 77 is within a normal range, but it's still slightly elevated compared to pre-surgery levels or your usual baseline. It's not uncommon for blood pressure to rise post-surgery, especially if you were already on blood pressure medications before the operation.

Given that you have just undergone hip repla

### Итог настройки prompt на debug

Улучшенный system prompt сделал ответы короче и осторожнее, однако не устранил фактические ошибки во всех случаях. Модель по-прежнему может ошибаться при интерпретации медицинских показателей и вводить неподтверждённые медицинские объяснения, несмотря на явные ограничения в инструкции.

Дальнейшая настройка на тех же примерах могла бы привести к подгонке под `debug`, поэтому prompt на этом этапе фиксируется и далее сравнивается с baseline на `dev`.

Этот результат показывает важное ограничение prompt engineering: инструкция может менять стиль и поведение модели, но не гарантирует исправление её фактических знаний.

## 8. Сравнение system prompts на dev

После настройки на `debug` оба system prompt фиксируются и сравниваются на новых примерах из `dev`.

Для сравнения используется один и тот же набор вопросов, одна модель и одинаковые параметры генерации. Единственная изменяемая часть — system prompt.

Поскольку генерация на всём `dev` вычислительно затратна, для выбора prompt используется фиксированная случайная подвыборка. Ответы сохраняются, чтобы эксперимент не требовалось пересчитывать.

In [37]:
dev_path = "../materials/dev.csv"
dev_df = pd.read_csv(dev_path)

print(dev_df.shape)
dev_df.head()

(1000, 2)


,input,output
0,my husband age is 29 yrs.height is 5.10ft.his ...,Hello Thanks for writing to your husbands seme...
1,My HBeAg is non-reactive but my Anti-HBc is re...,Hi dear thanks for asking question.... Noted y...
2,"hello Dr, The entire left side of my face has ...","Hi, in my opinion u should see a dentist at th..."
3,"Hi, I have slight pain in my lower right abdom...",HI ! Good afternoon. I am Chat Doctor answerin...
4,I am a physician investigating a workers compe...,"Hello, Yes, Lyme disease can cause Bells palsy..."


In [38]:
PROMPT_EVAL_SIZE = 50
SEED = 42

dev_for_prompt_eval = dev_df[
    ~dev_df["input"].isin(debug_df["input"])
].copy()

prompt_dev_df = (
    dev_for_prompt_eval
    .sample(n=PROMPT_EVAL_SIZE, random_state=SEED)
    .reset_index(drop=True)
)

prompt_dev_df["example_id"] = range(len(prompt_dev_df))

print("Dev без debug:", len(dev_for_prompt_eval))
print("Примеров для сравнения:", len(prompt_dev_df))

assert not prompt_dev_df["input"].isin(debug_df["input"]).any()

Dev без debug: 970
Примеров для сравнения: 50


### Генерация ответов

Для каждого вопроса генерируются два ответа:

- `original_answer` — Base Qwen с исходным system prompt;
- `improved_answer` — Base Qwen с улучшенным system prompt.

Порядок вопросов и параметры генерации для обоих вариантов одинаковы.

In [39]:
results = []

for i, row in prompt_dev_df.iterrows():
    original_answer = generate_answer(
        question=row["input"],
        system_prompt=ORIGINAL_SYSTEM_PROMPT,
        model=model,
        tokenizer=tokenizer,
    )

    improved_answer = generate_answer(
        question=row["input"],
        system_prompt=IMPROVED_SYSTEM_PROMPT,
        model=model,
        tokenizer=tokenizer,
    )

    results.append(
        {
            "example_id": row["example_id"],
            "question": row["input"],
            "reference": row["output"],
            "original_answer": original_answer,
            "improved_answer": improved_answer,
        }
    )

    print(f"{i + 1}/{len(prompt_dev_df)}")

1/50
2/50
3/50
4/50
5/50
6/50
7/50
8/50
9/50
10/50
11/50
12/50
13/50
14/50
15/50
16/50
17/50
18/50
19/50
20/50
21/50
22/50
23/50
24/50
25/50
26/50
27/50
28/50
29/50
30/50
31/50
32/50
33/50
34/50
35/50
36/50
37/50
38/50
39/50
40/50
41/50
42/50
43/50
44/50
45/50
46/50
47/50
48/50
49/50
50/50


In [40]:
prompt_results_df = pd.DataFrame(results)

prompt_results_df["original_tokens"] = (
    prompt_results_df["original_answer"]
    .apply(lambda x: len(tokenizer.encode(x, add_special_tokens=False)))
)

prompt_results_df["improved_tokens"] = (
    prompt_results_df["improved_answer"]
    .apply(lambda x: len(tokenizer.encode(x, add_special_tokens=False)))
)

print(prompt_results_df[["original_tokens", "improved_tokens"]].describe())

print(
    "Original >= 500:",
    (prompt_results_df["original_tokens"] >= 500).sum(),
)

print(
    "Improved >= 500:",
    (prompt_results_df["improved_tokens"] >= 500).sum(),
)

       original_tokens  improved_tokens
count         50.00000        50.000000
mean         341.02000       182.740000
std           92.33645        75.821167
min          183.00000        73.000000
25%          266.25000       125.000000
50%          324.00000       160.500000
75%          415.25000       246.250000
max          512.00000       340.000000
Original >= 500: 3
Improved >= 500: 0


In [41]:
results_path = "../results/prompt_dev_results.csv"
prompt_results_df.to_csv(results_path, index=False)

print("Сохранено:", results_path)

Сохранено: ../results/prompt_dev_results.csv


In [42]:
import random

rng = random.Random(SEED)

blind_rows = []

for _, row in prompt_results_df.iterrows():
    improved_on_left = rng.choice([True, False])

    if improved_on_left:
        answer_a = row["improved_answer"]
        answer_b = row["original_answer"]
        answer_a_source = "improved"
        answer_b_source = "original"
    else:
        answer_a = row["original_answer"]
        answer_b = row["improved_answer"]
        answer_a_source = "original"
        answer_b_source = "improved"

    blind_rows.append(
        {
            "example_id": row["example_id"],
            "question": row["question"],
            "answer_a": answer_a,
            "answer_b": answer_b,
            "answer_a_source": answer_a_source,
            "answer_b_source": answer_b_source,
        }
    )

blind_df = pd.DataFrame(blind_rows)

In [43]:
blind_eval_df = blind_df[
    ["example_id", "question", "answer_a", "answer_b"]
].copy()

blind_eval_df["winner"] = ""
blind_eval_df["comment"] = ""

blind_eval_df.to_csv(
    "../results/prompt_dev_blind_eval.csv",
    index=False,
)

blind_df[
    ["example_id", "answer_a_source", "answer_b_source"]
].to_csv(
    "../results/prompt_dev_blind_key.csv",
    index=False,
)

## 9. Слепое сравнение ответов

Чтобы не знать заранее, какой ответ получен каким system prompt, ответы оцениваются вслепую.

Для каждого вопроса показываются только два варианта — `Answer A` и `Answer B`. Информация о том, какому prompt соответствует каждый ответ, хранится отдельно и раскрывается только после завершения оценки.

Главный вопрос — какой ответ лучше подходит для медицинского ассистента. При сравнении учитываются:

1. фактическая корректность;
2. безопасность медицинских рекомендаций;
3. отсутствие неподтверждённых выводов;
4. релевантность вопросу;
5. адекватное выражение неопределённости;
6. ясность и отсутствие лишней информации.

Для каждой пары выбирается `A`, `B` или `Tie`.

In [44]:
evaluated_df = pd.read_csv(
    "../results/prompt_dev_blind_eval_filled.csv"
)

key_df = pd.read_csv(
    "../results/prompt_dev_blind_key.csv"
)

evaluation_df = evaluated_df.merge(
    key_df,
    on="example_id",
    validate="one_to_one",
)

def get_winner_prompt(row):
    if row["winner"] == "Tie":
        return "tie"

    if row["winner"] == "A":
        return row["answer_a_source"]

    if row["winner"] == "B":
        return row["answer_b_source"]

    raise ValueError(f"Unknown winner: {row['winner']}")

evaluation_df["winner_prompt"] = evaluation_df.apply(
    get_winner_prompt,
    axis=1,
)

print(evaluation_df["winner_prompt"].value_counts())
print(
    evaluation_df["winner_prompt"]
    .value_counts(normalize=True)
    .mul(100)
    .round(1)
)

winner_prompt
improved    36
original    14
Name: count, dtype: int64
winner_prompt
improved    72.0
original    28.0
Name: proportion, dtype: float64


### Результат сравнения

После раскрытия ключа improved system prompt оказался предпочтительнее в 36 из 50 случаев (72%), тогда как исходный prompt — в 14 случаях (28%).

Преимущество improved prompt сохранялось независимо от того, находился ли его ответ в позиции `A` или `B`, поэтому случайный дисбаланс позиций не оказал заметного влияния на результат.

Ручной анализ показал, что улучшенный prompt чаще давал более осторожные, релевантные и безопасные ответы и реже вводил неподтверждённые диагнозы, обследования или лечение.

При этом фактические ошибки полностью не исчезли. Следовательно, prompt engineering заметно улучшает поведение модели, но не решает проблему медицинской достоверности сам по себе.